In [1]:
from math import ceil
from typing import cast

import torch
from torch import nn
from torch.utils.data import DataLoader

from fun.data.ct_dataset import CTPostProcessDataset
from fun.data.ellipses_dataset import EllipsesDataset
from fun.models.interp_unet import InterpolatingUNet

In [4]:
from fun.models.interp_unet import InterpolatingConv2d


weights = torch.load("../runs/UNet-64x64-11/weights/best-all.pt", map_location="cpu")
model = InterpolatingUNet(1, 1, base_input_size=64, max_scale_factor=4, upscale_weights=True, three_initial_convs=True)

param_name_map = {}
for name, _ in model.named_parameters():
    param_name_map[name] = input(name + ": ")
    if param_name_map[name] == "":
        param_name_map[name] = name

In [ ]:
param_name_map = {
    "_down_blocks.0.0.weight": "_down_blocks.0.0.weight",
    "_down_blocks.0.0.bias": "_down_blocks.0.0.bias",
    "_down_blocks.0.2.weight": "_down_blocks.0.2.weight",
    "_down_blocks.0.2.bias": "_down_blocks.0.2.bias",
    "_down_blocks.0.4.weight": "_down_blocks.0.4.weight",
    "_down_blocks.0.4.bias": "_down_blocks.0.4.bias",
    "_down_blocks.1.1.weight": "_down_blocks.1.1.weight",
    "_down_blocks.1.1.bias": "_down_blocks.1.1.bias",
    "_down_blocks.1.3.weight": "_down_blocks.1.3.weight",
    "_down_blocks.1.3.bias": "_down_blocks.1.3.bias",
    "_down_blocks.2.1.weight": "_down_blocks.2.1.weight",
    "_down_blocks.2.1.bias": "_down_blocks.2.1.bias",
    "_down_blocks.2.3.weight": "_down_blocks.2.3.weight",
    "_down_blocks.2.3.bias": "_down_blocks.2.3.bias",
    "_down_blocks.3.1.weight": "_down_blocks.3.1.weight",
    "_down_blocks.3.1.bias": "_down_blocks.3.1.bias",
    "_down_blocks.3.3.weight": "_down_blocks.3.3.weight",
    "_down_blocks.3.3.bias": "_down_blocks.3.3.bias",
    "_central_block.1.weight": "_central_block.1.weight",
    "_central_block.1.bias": "_central_block.1.bias",
    "_central_block.3.weight": "_central_block.3.weight",
    "_central_block.3.bias": "_central_block.3.bias",
    "_central_block.6.weight": None,
    "_central_block.6.bias": None,
    "_up_blocks.0.0.weight": "_up_blocks.0.0.weight",
    "_up_blocks.0.0.bias": "_up_blocks.0.0.bias",
    "_up_blocks.0.2.weight": "_up_blocks.0.2.weight",
    "_up_blocks.0.2.bias": "_up_blocks.0.2.bias",
    "_up_blocks.0.5.weight": None,
    "_up_blocks.0.5.bias": None,
    "_up_blocks.1.0.weight": "_up_blocks.1.0.weight",
    "_up_blocks.1.0.bias": "_up_blocks.1.0.bias",
    "_up_blocks.1.2.weight": "_up_blocks.1.2.weight",
    "_up_blocks.1.2.bias": "_up_blocks.1.2.bias",
    "_up_blocks.1.5.weight": None,
    "_up_blocks.1.5.bias": None,
    "_up_blocks.2.0.weight": "_up_blocks.2.0.weight",
    "_up_blocks.2.0.bias": "_up_blocks.2.0.bias",
    "_up_blocks.2.2.weight": "_up_blocks.2.2.weight",
    "_up_blocks.2.2.bias": "_up_blocks.2.2.bias",
    "_up_blocks.2.5.weight": None,
    "_up_blocks.2.5.bias": None,
    "_up_blocks.3.0.weight": "_up_blocks.3.0.weight",
    "_up_blocks.3.0.bias": "_up_blocks.3.0.bias",
    "_up_blocks.3.2.weight": "_up_blocks.3.2.weight",
    "_up_blocks.3.2.bias": "_up_blocks.3.2.bias",
    "_up_blocks.3.4.weight": "_up_blocks.3.4.weight",
    "_up_blocks.3.4.bias": "_up_blocks.3.4.bias"
}

{'_down_blocks.0.0.weight': '_down_blocks.0.0.weight',
 '_down_blocks.0.0.bias': '_down_blocks.0.0.bias',
 '_down_blocks.0.2.weight': '_down_blocks.0.2.weight',
 '_down_blocks.0.2.bias': '_down_blocks.0.2.bias',
 '_down_blocks.0.4.weight': '_down_blocks.0.4.weight',
 '_down_blocks.0.4.bias': '_down_blocks.0.4.bias',
 '_down_blocks.1.1.weight': '_down_blocks.1.1.weight',
 '_down_blocks.1.1.bias': '_down_blocks.1.1.bias',
 '_down_blocks.1.3.weight': '_down_blocks.1.3.weight',
 '_down_blocks.1.3.bias': '_down_blocks.1.3.bias',
 '_down_blocks.2.1.weight': '_down_blocks.2.1.weight',
 '_down_blocks.2.1.bias': '_down_blocks.2.1.bias',
 '_down_blocks.2.3.weight': '_down_blocks.2.3.weight',
 '_down_blocks.2.3.bias': '_down_blocks.2.3.bias',
 '_down_blocks.3.1.weight': '_down_blocks.3.1.weight',
 '_down_blocks.3.1.bias': '_down_blocks.3.1.bias',
 '_down_blocks.3.3.weight': '_down_blocks.3.3.weight',
 '_down_blocks.3.3.bias': '_down_blocks.3.3.bias',
 '_central_block.1.weight': '_central_block.1.

In [ ]:
test_datasets = {
    f"{r}x{r}": CTPostProcessDataset(
        torch.utils.data.Subset(EllipsesDataset.from_file("../data/test.h5"), range(200)),
        angles=torch.linspace(0.0, torch.pi * 0.75, ceil(4 * r * 0.75)),
        pos_count=2 * r,
        target_shape=(r, r),
        noise_type="gaussian",
        noise_level=0.0,
    )
    for r in [x for i in range(4, 10) for x in [2**i, 2**i + 2 ** (i - 2) + 1, 2**i + 2 ** (i - 1), 2 ** (i + 1) - 2 ** (i - 2) - 1, 2**i + 1]]
}
test_dataloaders = {
    name: DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )
    for name, dataset in test_datasets.items()
}

In [ ]:
DEVICE = "cpu" if not torch.cuda.is_available() else "cuda"

loss_function = nn.MSELoss()
model.to(DEVICE)
for name, dataloader in test_dataloaders.items():
    try:
        loss = 0.0
        for batch in dataloader:
            input_ = batch["input"].to(DEVICE)
            target = batch["target"].to(DEVICE)
            prediction = model(input_)
            loss += loss_function(prediction, target).item()
        loss /= len(dataloader)
    except Exception as e:
        print(f"Error testing {name} dataset: {e}")
        continue
    print(f"Finished testing {name} dataset: {loss}\n")